# Music Streaming Behavior — Springfield vs Shelbyville

**Author:** Mario Alberto Noriega Barrios  
**Tools:** Python, pandas, Jupyter Notebook

---

## Business question

A music streaming service wants to know whether its users listen differently depending on **the city** (Springfield or Shelbyville) and **the day of the week** (Monday or Friday).

**Working hypothesis:** listening activity differs between the two cities and between Monday and Friday.

## Dataset

65,079 plays logged by the service, one row per play.

| Column | Description |
|---|---|
| `user_id` | User identifier |
| `track` | Song title |
| `artist` | Artist name |
| `genre` | Genre of the song |
| `city` | City of the user |
| `time` | Time of the play |
| `day` | Day of the week |

---
## 1. Data review

In [1]:
import pandas as pd

df = pd.read_csv("music_project_en.csv")
df.head(10)

,userID,Track,artist,genre,City,time,Day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
5,842029A1,Chains,Obladaet,rusrap,Shelbyville,13:09:41,Friday
6,4CB90AA5,True,Roman Messer,dance,Springfield,13:00:07,Wednesday
7,F03E1C1F,Feeling This Way,Polina Griffith,dance,Springfield,20:47:49,Wednesday
8,8FA1D3BE,L’estate,Julia Dalia,ruspop,Springfield,09:17:40,Friday
9,E772D5C0,Pessimist,NaN,dance,Shelbyville,21:20:49,Wednesday


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0     userID  65079 non-null  str  
 1   Track     63736 non-null  str  
 2   artist    57512 non-null  str  
 3   genre     63881 non-null  str  
 4     City    65079 non-null  str  
 5   time      65079 non-null  str  
 6   Day       65079 non-null  str  
dtypes: str(7)
memory usage: 3.5 MB


**What the review shows:**

- 65,079 rows and 7 columns, all stored as text. That includes `time`, which should be a time and is stored as a string. It is not needed for this analysis, so it is left as is.
- The dataset has the two columns the hypothesis needs (`city` and `day`) and enough rows in every city and day combination, so it is sufficient to test the hypothesis.
- Header problems: mixed upper and lower case, and extra spaces in `userID` and `City`.
- Missing values in `track` (1,343), `artist` (7,567) and `genre` (1,198).
- 3,826 fully duplicated rows.

---
## 2. Data cleaning

### 2.1 Headers

Lowercase, no extra spaces, and `snake_case` for `user_id`.

In [3]:
df.columns = [col.strip().lower() for col in df.columns]
df = df.rename(columns={"userid": "user_id"})
df.columns

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')

### 2.2 Missing values

`track`, `artist` and `genre` are descriptive fields and are not used to count plays. Rows with a missing value still carry a valid user, city and day, so they are **kept** and filled with `unknown` instead of being deleted. Deleting them would remove plays from the counts.

In [4]:
print(df.isna().sum())

user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64


In [5]:
columns_to_fill = ["track", "artist", "genre"]

for column in columns_to_fill:
    df[column] = df[column].fillna("unknown")

print(df.isna().sum())

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


### 2.3 Explicit duplicates

In [6]:
print("Duplicates before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates after:", df.duplicated().sum())
print("Rows remaining:", len(df))

Duplicates before: 3826
Duplicates after: 0
Rows remaining: 61253


### 2.4 Implicit duplicates in `genre`

The same genre appears written in different ways (`hip`, `hop`, `hip-hop`, `hiphop`). `triphop` is a different genre and is left untouched.

In [7]:
print([g for g in sorted(df["genre"].unique()) if "hip" in g or "hop" in g])

['hip', 'hip-hop', 'hiphop', 'hop', 'triphop']


In [8]:
def replace_wrong_values(df, column, wrong_values, correct_value):
    """
    Replaces misspelled or inconsistent values in a column with a single correct value.

    Args:
        df (DataFrame): DataFrame to modify.
        column (str): Name of the column to fix.
        wrong_values (list): Values that should be replaced.
        correct_value (str): Value that replaces them.

    Returns:
        DataFrame: The DataFrame with the column corrected.
    """
    for wrong_value in wrong_values:
        df[column] = df[column].replace(wrong_value, correct_value)
    return df

The function is checked before and after it runs, so the result does not depend on reading a list of 268 genres.

In [9]:
wrong_genres = ["hip", "hop", "hip-hop"]

print("Rows with a wrong variant before:", df["genre"].isin(wrong_genres).sum())

df = replace_wrong_values(df, "genre", wrong_genres, "hiphop")

print("Rows with a wrong variant after: ", df["genre"].isin(wrong_genres).sum())
print("Rows labeled hiphop now:         ", (df["genre"] == "hiphop").sum())

Rows with a wrong variant before: 2977
Rows with a wrong variant after:  0
Rows labeled hiphop now:          3056


**Known limitation:** the genre list also contains one corrupted value (`ïîï`), caused by a text encoding problem in the source file. It affects one row and was left as is.

---
## 3. Analysis

### 3.1 Plays and users by city

Plays are not the same as people. A city can have more plays simply because it has more users, so both are counted, along with plays per user.

In [10]:
by_city = df.groupby("city").agg(
    plays=("track", "count"),
    users=("user_id", "nunique"),
)
by_city["plays_per_user"] = (by_city["plays"] / by_city["users"]).round(2)
by_city

,plays,users,plays_per_user
city,,,
Shelbyville,18512,12423,1.49
Springfield,42741,29358,1.46


Springfield has more than twice the plays of Shelbyville (42,741 against 18,512), but it also has more than twice the users (29,358 against 12,423). Per user the difference almost disappears: **1.46 plays in Springfield and 1.49 in Shelbyville**. The gap in volume comes from the size of the population, not from users listening more.

### 3.2 Plays by day

In [11]:
by_day = df.groupby("day")["track"].count().loc[["Monday", "Friday"]]
print(by_day)
print(f"\nFriday vs Monday: {(by_day['Friday'] / by_day['Monday'] - 1):+.1%}")

day
Monday    21354
Friday    21840
Name: track, dtype: int64



Friday vs Monday: +2.3%


Friday has slightly more plays than Monday (21,840 against 21,354). The difference is about 2.3%, which is small.

### 3.3 City and day combined

`number_tracks()` counts plays for a given day and city.

In [12]:
def number_tracks(day, city):
    """
    Counts the plays registered in a given city on a given day.

    Args:
        day (str): Day of the week, for example 'Monday'.
        city (str): City name, for example 'Springfield'.

    Returns:
        int: Number of plays in that city on that day.
    """
    df_day = df[df["day"] == day]
    df_city = df_day[df_day["city"] == city]
    return df_city["user_id"].count()

In [13]:
results = {
    (day, city): number_tracks(day, city)
    for day in ["Monday", "Friday"]
    for city in ["Springfield", "Shelbyville"]
}

summary = pd.Series(results).unstack()
summary["Total"] = summary.sum(axis=1)
summary

,Shelbyville,Springfield,Total
Friday,5895,15945,21840
Monday,5614,15740,21354


The four results are checked against the totals from the previous section, so the function and the groupby agree.

In [14]:
assert summary["Total"]["Monday"] == by_day["Monday"]
assert summary["Total"]["Friday"] == by_day["Friday"]

change = (summary.loc["Friday", ["Springfield", "Shelbyville"]]
          / summary.loc["Monday", ["Springfield", "Shelbyville"]] - 1)
print("Change from Monday to Friday:")
print(change.map("{:+.1%}".format))

Change from Monday to Friday:
Springfield    +1.3%
Shelbyville    +5.0%
dtype: str


---
## 4. Conclusions

**Answer to the question.** The data does not show that users behave differently by city, and the day of the week has only a small effect.

- **City:** it changes the *volume* of plays, not the *habit*. Springfield has more plays because it has more users. Per user the figure is almost identical (1.46 against 1.49).
- **Day:** Friday has 2.3% more plays than Monday. The change is small in Springfield (+1.3%) and larger in Shelbyville (+5.0%), so if there is a day effect, it is stronger in the smaller city.

**Data quality.** The raw data had inconsistent headers, missing values in three columns, 3,826 duplicated rows and several spellings of the same genre. Missing text values were filled with `unknown` to keep the plays in the count, duplicates were removed, and genre variants were unified and verified. Without this, the plays per city and day would have been inflated by the duplicates.

**Limitations and next steps.**
- Only two days of the week were analyzed. A full week would show whether Monday and Friday are representative.
- The data counts plays, not listening time, so a long song and a short one weigh the same.
- Whether the +5.0% in Shelbyville is a real pattern or normal variation would need a statistical test, which is the natural next step for this analysis.